<a href="https://colab.research.google.com/github/TrackTech/AgenticAI/blob/main/Devops_ReAct_OpenAPI_function_tooling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **IT Support Agent**

This is a coding assignment for the **Level 1 IT Incident Responder**.


**Objective:** The workflow acts as first responder to know issue with the servers. It:

1. **Investigate:** Check server health and logs when a user reports an issue.
2. **Act:** If CPU is critical (>90%), it should **Restart** the service.
3. **Escalate:** If the issue is complex or logs show a major problem it should **Escalate** to a human.

In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [2]:
client = OpenAI(api_key=userdata.get('open_api_key'))

# Initialize Client

==========================================
## PART 1: DEFINE THE TOOLS (BUSINESS LOGIC)
==========================================

In [11]:
def get_server_health(server_id: str) -> str:
    """Returns CPU and Memory usage for a given server."""
    print(f"-> TOOL: Checking health for {server_id}...")

    metrics = {
        # Scenario 1: High CPU (Needs Restart)
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},

        # Scenario 2: Healthy (No Action Needed)
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},

        # Scenario 3: High Memory Leak (Needs Restart or Escalation)
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},

        # Scenario 4: Network/Dependency Failure (Needs Escalation)
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},

        # Scenario 5: Completely Normal
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }

    result = metrics.get(server_id, {"error": "Server not found. Check the ID."})
    return json.dumps(result)


In [12]:
def fetch_recent_logs(server_id: str, lines: int = 5) -> str:
    """Returns the last N lines of logs."""
    print(f"-> TOOL: Fetching last {lines} log lines for {server_id}...")

    # Different logs for different servers to trigger different agent behaviors
    log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }

    # Default logs if server not found in specific list
    default_logs = ["[INFO] System stable", "[INFO] Heartbeat signal received"]

    logs = log_database.get(server_id, default_logs)
    return json.dumps({"logs": logs[:lines]})

In [34]:
def restart_service(server_id: str) -> str:
    """
    Mocking a actual server restart
    Use slack web hook to send a message to dev ops group
    """
    output = '{"status":"success","server_id":"'+server_id+'","message":"Server restarted successfully"}'

    return json.dumps(output)


In [14]:
def escalate_to_engineer(summary: str) -> str:
    """
    Likely call pager duty or a slack web hook
    """
    print("TOOL: Escalating to human...")
    msg = "Ticket has been created, summary is "+summary
    return '{"status":"success","ticket_id":"1234","message":msg}'


In [35]:
# Map functions for the agent execution loop
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "fetch_recent_logs": fetch_recent_logs,
    "restart_service": restart_service,
    "escalate_to_engineer": escalate_to_engineer,
}

==========================================
## PART 2: DEFINE THE AGENT SCHEMA
==========================================


In [36]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_server_health",
            "description": "Checks the current CPU and memory usage of a specific server.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server, e.g., 'payment-server-01'"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Retrieves the most recent log entries from a server to diagnose errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."},
                    "lines": {"type": "integer", "description": "Number of log lines to fetch."}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "Restart server as CPU>90%",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id":{"type":"string","description":"The ID of the server to restart."}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Escalates the issue to a human engineer when automated fixes fail or the error is unknown or logs are indicating no issues.",
            "parameters": {
                "type": "object",
                "properties": {
                     "summary":{"type":"string","description":"Summary of the reason why escalated to engineer"}
                },
                "required": ["summary"]
            }
        }
    }
]


 ==========================================
## PART 3: THE AGENT EXECUTION LOOP
 ==========================================


In [37]:
def run_it_agent(user_issue: str):
    print(f"\n--- New Incident: {user_issue} ---")

    messages = [
        {"role": "system", "content": "You are a Level 1 IT Responder. Investigate server issues. "
                                      "If CPU or Memory is > 90%, restart the service. If logs show critical dependency errors (like connection refused) that a restart won't fix, escalate to an engineer."},
        {"role": "user", "content": user_issue}
    ]

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            # model="gpt-4.1-mini",
            # model = "gpt-5",
            model = "gpt-4o-mini",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        print(f"how many choices did i have-{len(response.choices)}, and message - {response_msg}")
        messages.append(response_msg)

        if response_msg.tool_calls:
            for tool_call in response_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # Retrieve the actual python function based on name
                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)

                if function_to_call:
                    # Execute the function
                    tool_output = function_to_call(**func_args)

                    messages.append({
                        "role":"tool",
                        "tool_call_id":tool_call.id,
                        "name":func_name,
                        "content":tool_output

                    })
                    print(f"Tool call id = {tool_call.id}")

        else:
            print(f"\n[FINAL RESPONSE]: {response_msg.content}")
            for msg in messages:
              role = getattr(msg, 'role', None) or msg.get('role')
              print(f"Message role: {role}")
            break

 ==========================================
## PART 4: TEST SCENARIOS
 ==========================================


In [38]:
# Scenario A: Should trigger a restart (CPU is 98%)
run_it_agent("The payment-server-01 is extremely slow and timing out.")


--- New Incident: The payment-server-01 is extremely slow and timing out. ---

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_eQdqtXtkFBodG1Djd3bw3whH', function=Function(arguments='{"server_id":"payment-server-01"}', name='get_server_health'), type='function')])
-> TOOL: Checking health for payment-server-01...
Tool call id = call_eQdqtXtkFBodG1Djd3bw3whH

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_kWmL0y3rrJCwjpfNXfLmqya6', function=Function(arguments='{"server_id":"payment-server-01"}', name='restart_service'), type='function')])
Tool call id = call_kWmL0y3rrJCwjpfNXfLmqya6

[AI Thinking...]
how many choi

In [39]:
# Scenario B: Should trigger an escalation (DB is healthy but logs might be weird)
run_it_agent("Something is wrong with db-node-02")


--- New Incident: Something is wrong with db-node-02 ---

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_1aiLznMGzAySpGKzoUjYwefp', function=Function(arguments='{"server_id": "db-node-02"}', name='get_server_health'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_RmcfN4mYfw3P7bhMs3FeY7EG', function=Function(arguments='{"server_id": "db-node-02", "lines": 50}', name='fetch_recent_logs'), type='function')])
-> TOOL: Checking health for db-node-02...
Tool call id = call_1aiLznMGzAySpGKzoUjYwefp
-> TOOL: Fetching last 50 log lines for db-node-02...
Tool call id = call_RmcfN4mYfw3P7bhMs3FeY7EG

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content='The health check for **db-node-02** indicates that both CPU (12%) and Memory (60%) are within health

In [40]:
# Scenario C: The High Memory Case (auth-service-03)
# Agent should see Memory 95% + OutOfMemoryError logs -> Restart
run_it_agent("Users are reporting login failures on auth-service-03.")

print("\n" + "="*50 + "\n")


--- New Incident: Users are reporting login failures on auth-service-03. ---

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_AWKKFV80hRByb4DyRC1dOPzs', function=Function(arguments='{"server_id": "auth-service-03"}', name='get_server_health'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_fmtU7vSEtsFL0YdxGG2OFDsI', function=Function(arguments='{"server_id": "auth-service-03", "lines": 50}', name='fetch_recent_logs'), type='function')])
-> TOOL: Checking health for auth-service-03...
Tool call id = call_AWKKFV80hRByb4DyRC1dOPzs
-> TOOL: Fetching last 50 log lines for auth-service-03...
Tool call id = call_fmtU7vSEtsFL0YdxGG2OFDsI

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=N

In [41]:
# Scenario D: The Dependency Failure (search-index-09)
# Agent should see healthy CPU but "Connection Refused" logs -> Escalate
run_it_agent("Search isn't working. Can you check search-index-09?")

print("\n" + "="*50 + "\n")


--- New Incident: Search isn't working. Can you check search-index-09? ---

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_HGaokD6gIx3HAAUPeFuOqkZ6', function=Function(arguments='{"server_id": "search-index-09"}', name='get_server_health'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_txsrDA1qRsKZB2ZmiviakJfP', function=Function(arguments='{"server_id": "search-index-09", "lines": 50}', name='fetch_recent_logs'), type='function')])
-> TOOL: Checking health for search-index-09...
Tool call id = call_HGaokD6gIx3HAAUPeFuOqkZ6
-> TOOL: Fetching last 50 log lines for search-index-09...
Tool call id = call_txsrDA1qRsKZB2ZmiviakJfP

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=Non

In [42]:
# Scenario E: The Healthy Server (frontend-node-04)
# Agent should see normal stats and 200 OK logs -> Do nothing / Report healthy
run_it_agent("Check frontend-node-04 just to be safe.")


--- New Incident: Check frontend-node-04 just to be safe. ---

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_nFeKMyzvLRkW9U7noYNoS35f', function=Function(arguments='{"server_id": "frontend-node-04"}', name='get_server_health'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_nG3VY3jjHbLazYpG4Oe38EaK', function=Function(arguments='{"server_id": "frontend-node-04", "lines": 50}', name='fetch_recent_logs'), type='function')])
-> TOOL: Checking health for frontend-node-04...
Tool call id = call_nFeKMyzvLRkW9U7noYNoS35f
-> TOOL: Fetching last 50 log lines for frontend-node-04...
Tool call id = call_nG3VY3jjHbLazYpG4Oe38EaK

[AI Thinking...]
how many choices did i have-1, and message - ChatCompletionMessage(content='The server **frontend-node-04** is currently healthy with the following